In [ ]:
#U-NET
import os
import torch
import torch.nn as nn
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from tqdm import tqdm
from PIL import Image
import torch.nn.functional as F

# ======================
# 路径配置
# ======================
DATA_PATH = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp_weekly_with_monthly_pred_with_elev.nc"
MODEL_PATH = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/PR/U-NET/subseasonal_model_u-tp_best.pth"
NORM_PARAM_FILE = '/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/PR/U-NET/norm_params_tp22.npz'
OUTPUT_NC = "pred_subseasonal_model_u_tp_20260831.nc"

# ======================
# 输入输出变量
# ======================
input_vars = [f'tp_hist_{i}' for i in range(20)] + \
             [f'z200_hist_{i}' for i in range(10)] + \
             [f'z200_z300_hist_{i}' for i in range(10)] + \
             [f'z200_z500_hist_{i}' for i in range(10)] + \
             [f'q700_hist_{i}' for i in range(10)] + \
             [f'cloudfraction800_hist_{i}' for i in range(10)] + \
             ['pred_tp_month', 'elevation']

output_vars = ['tp_mon', 'tp_wed', 'tp_fri', 'tp_sun', 
               'tp_tue_next', 'tp_thu_next', 'tp_sat_next']

# ======================
# 加载数据
# ======================
ds = xr.open_dataset(DATA_PATH)

# 检查并填充 NaN
for var in input_vars:
    if ds[var].isnull().any():
        print(f"[NaN Warning] 变量 {var} 中存在 NaN，将填充为 0")
        ds[var] = ds[var].fillna(0)

# ======================
# 加载标准化参数
# ======================
params = np.load(NORM_PARAM_FILE)
x_mean, x_std, y_mean, y_std = params['x_mean'], params['x_std'], params['y_mean'], params['y_std']

# ======================
# 数据预处理函数
# ======================
def prepare_input(ds):
    x_data = ds[input_vars].to_array().transpose('time', 'variable', 'latitude', 'longitude')
    x_norm = (x_data - x_mean[:, None, None]) / x_std[:, None, None]
    return np.nan_to_num(x_norm.values.astype(np.float32))

X = prepare_input(ds)

# ======================
# 模型定义
# ======================
def center_crop(tensor, target_size):
    _, _, h, w = tensor.size()
    th, tw = target_size
    x1 = (h - th) // 2
    y1 = (w - tw) // 2
    return tensor[:, :, x1:x1+th, y1:y1+tw]

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.residual = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()

    def forward(self, x):
        res = self.residual(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += res
        return self.relu(out)

class UNet_Res(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.enc1 = ResidualBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ResidualBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ResidualBlock(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = ResidualBlock(256, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = ResidualBlock(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ResidualBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ResidualBlock(128, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        e3 = self.enc3(p2)
        p3 = self.pool3(e3)
        b = self.bottleneck(p3)

        u3 = self.up3(b)
        if u3.size()[2:] != e3.size()[2:]:
            e3 = center_crop(e3, u3.size()[2:])
        d3 = self.dec3(torch.cat([u3, e3], dim=1))

        u2 = self.up2(d3)
        if u2.size()[2:] != e2.size()[2:]:
            e2 = center_crop(e2, u2.size()[2:])
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        if u1.size()[2:] != e1.size()[2:]:
            e1 = center_crop(e1, u1.size()[2:])
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        out = self.output_layer(d1)
        if out.shape[2:] != (121, 240):
            out = F.interpolate(out, size=(121, 240), mode='bilinear', align_corners=False)
        return out

# ======================
# 加载模型
# ======================
device = torch.device("cpu")
model = UNet_Res(len(input_vars), len(output_vars)).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

# ======================
# 执行推理
# ======================
Y_pred = []
with torch.no_grad():
    for i in tqdm(range(X.shape[0]), desc="Predicting"):
        x = torch.from_numpy(X[i:i+1]).to(device)
        pred = model(x).cpu().numpy()
        pred_denorm = pred * y_std[:, None, None] + y_mean[:, None, None]
        Y_pred.append(pred_denorm[0])
Y_pred = np.stack(Y_pred, axis=0)

# ======================
# 保存 NetCDF 文件
# ======================
output_ds = xr.Dataset(
    {var: (("time", "latitude", "longitude"), Y_pred[:, i]) 
     for i, var in enumerate(output_vars)},
    coords={
        "time": ds.time.values,
        "latitude": ds.latitude.values,
        "longitude": ds.longitude.values,
    }
)
output_ds.to_netcdf(OUTPUT_NC)
print(f"✅ Saved prediction to {OUTPUT_NC}")


In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 脚本/pred_subseasonal_model_u_tp_20260831.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
print(ds.variables)
# 如果需要查看数据集的维度，可以使用
print(ds.dims)

In [ ]:
# UPU-NET
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import torch
import torch.nn as nn
import torch.nn.functional as F
import xarray as xr
import numpy as np
from tqdm import tqdm

# ===========================
# 路径配置
# ===========================
INPUT_FILE = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp_weekly_with_monthly_pred_with_elev.nc"
OUTPUT_FILE = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 脚本/33subseasonal_model_u-tp_best_acc_20260831.nc"
MODEL_PATH = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/PR/UPU-NET/33subseasonal_model_u-tp_best_acc.pth"
NORM_PARAM_FILE = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/PR/UPU-NET/33norm_params_tp22.npz"

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
DEVICE = torch.device("cpu")

# ===========================
# 变量定义（与训练严格一致）
# ===========================
input_vars = (
    [f"tp_hist_{i}" for i in range(20)] +
    [f"z200_hist_{i}" for i in range(10)] +
    [f"z200_z300_hist_{i}" for i in range(10)] +
    [f"z200_z500_hist_{i}" for i in range(10)] +
    [f"q700_hist_{i}" for i in range(10)] +
    [f"cloudfraction800_hist_{i}" for i in range(10)] +
    ["pred_tp_month", "elevation"]
)

target_vars = [
    "tp_mon", "tp_wed", "tp_fri", "tp_sun",
    "tp_tue_next", "tp_thu_next", "tp_sat_next"
]

# ===========================
# 读取归一化参数
# ===========================
norm = np.load(NORM_PARAM_FILE)
x_mean = norm["x_mean"]
x_std = norm["x_std"]
y_mean = norm["y_mean"]
y_std = norm["y_std"]

# ===========================
# 数据准备函数
# ===========================
def prepare_input(ds):
    x = (
        ds[input_vars]
        .to_array()
        .transpose("time", "variable", "latitude", "longitude")
        .values
        .astype(np.float32)
    )
    x = (x - x_mean[:, None, None]) / x_std[:, None, None]
    return x

# ===========================
# 模型结构
# ===========================
class SEBlock(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c, c // r, 1),
            nn.ReLU(),
            nn.Conv2d(c // r, c, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c)
        )
        self.skip = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()
        self.se = SEBlock(out_c)

    def forward(self, x):
        return F.relu(self.se(self.conv(x) + self.skip(x)))

def center_crop(t, h, w):
    _, _, H, W = t.shape
    dh = (H - h) // 2
    dw = (W - w) // 2
    return t[:, :, dh:dh+h, dw:dw+w]

class UNet_MultiHead(nn.Module):
    def __init__(self, in_c, out_vars):
        super().__init__()
        self.enc1 = ResidualBlock(in_c, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.enc3 = ResidualBlock(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ResidualBlock(256, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = ResidualBlock(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = ResidualBlock(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = ResidualBlock(128, 64)

        self.heads = nn.ModuleList([nn.Conv2d(64, 1, 1) for _ in range(out_vars)])

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))

        u3 = self.up3(b)
        d3 = self.dec3(torch.cat([u3, center_crop(e3, *u3.shape[2:])], 1))
        u2 = self.up2(d3)
        d2 = self.dec2(torch.cat([u2, center_crop(e2, *u2.shape[2:])], 1))
        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, center_crop(e1, *u1.shape[2:])], 1))

        out = torch.cat([h(d1) for h in self.heads], dim=1)
        return F.interpolate(out, size=(121, 240), mode="bilinear", align_corners=False)

# ===========================
# 加载模型
# ===========================
model = UNet_MultiHead(len(input_vars), len(target_vars)).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# ===========================
# 推理单个文件
# ===========================
print(f"\n🚀 Predicting {INPUT_FILE} ...")
ds = xr.open_dataset(INPUT_FILE)

# NaN 保险
for v in input_vars:
    if ds[v].isnull().any():
        ds[v] = ds[v].fillna(0)

X = prepare_input(ds)

preds = []
with torch.no_grad():
    for t in tqdm(range(X.shape[0]), desc="timesteps"):
        x = torch.from_numpy(X[t:t+1]).float().to(DEVICE)
        y = model(x).cpu().numpy()[0]
        preds.append(y)

preds = np.stack(preds, axis=0)
preds = preds * y_std[:, None, None] + y_mean[:, None, None]

ds_out = xr.Dataset(
    {v: (("time", "latitude", "longitude"), preds[:, i])
     for i, v in enumerate(target_vars)},
    coords={
        "time": ds.time.values,
        "latitude": ds.latitude.values,
        "longitude": ds.longitude.values,
    }
)

ds_out.to_netcdf(OUTPUT_FILE)
print(f"✅ Saved {OUTPUT_FILE}")


In [ ]:
# SA
import os
import torch
import torch.nn as nn
import xarray as xr
import numpy as np
from tqdm import tqdm

# ======================
# 路径配置
# ======================
DATA_FILE = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp-1/tp_weekly_with_monthly_pred_with_elev.nc"
MODEL_PATH = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/PR/SA/33best_ACC_model.pth"
OUT_FILE = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 脚本/33SpatialAttentionResNet_TP_20260831.nc"
os.makedirs(os.path.dirname(OUT_FILE), exist_ok=True)

DEVICE = torch.device("cpu")

# ======================
# 输入 / 输出变量
# ======================
input_vars = (
    [f'tp_hist_{i}' for i in range(20)] +
    [f'z200_hist_{i}' for i in range(10)] +
    [f'z200_z300_hist_{i}' for i in range(10)] +
    [f'z200_z500_hist_{i}' for i in range(10)] +
    [f'q700_hist_{i}' for i in range(10)] +
    [f'cloudfraction800_hist_{i}' for i in range(10)] +
    ['pred_tp_month', 'elevation']
)

output_vars = [
    'tp_mon', 'tp_wed', 'tp_fri', 'tp_sun',
    'tp_tue_next', 'tp_thu_next', 'tp_sat_next'
]

VAR_GROUPS = {
    "tp": list(range(0, 20)),
    "gh": list(range(20, 50)),
    "q":  list(range(50, 60)),
    "cloud": list(range(60, 70)),
    "static": list(range(70, len(input_vars)))
}

# ======================
# 模型定义（原封不动）
# ======================
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attn = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return x * attn

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.attn = SpatialAttention()
        self.skip = nn.Conv2d(in_c, out_c, 1) if in_c != out_c else nn.Identity()

    def forward(self, x):
        out = self.relu(self.conv1(x))
        out = self.conv2(out)
        out = self.attn(out)
        return self.relu(out + self.skip(x))

class VariableEncoder(nn.Module):
    def __init__(self, in_c, out_c=32):
        super().__init__()
        self.block = nn.Sequential(
            ResidualBlock(in_c, out_c),
            ResidualBlock(out_c, out_c)
        )

    def forward(self, x):
        return self.block(x)

class TimeStepDecoder(nn.Module):
    def __init__(self, in_c):
        super().__init__()
        self.decoder = nn.Sequential(
            ResidualBlock(in_c, 64),
            ResidualBlock(64, 32),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, x):
        return self.decoder(x)

class SpatialAttentionResNet(nn.Module):
    def __init__(self, input_vars, target_vars):
        super().__init__()
        self.encoders = nn.ModuleDict({
            name: VariableEncoder(len(idxs))
            for name, idxs in VAR_GROUPS.items()
        })

        fusion_c = 32 * len(VAR_GROUPS)
        self.fusion = nn.Sequential(
            ResidualBlock(fusion_c, 128),
            ResidualBlock(128, 64)
        )

        self.decoders = nn.ModuleDict({
            name: TimeStepDecoder(64)
            for name in target_vars
        })

    def forward(self, x):
        feats = []
        for name, idxs in VAR_GROUPS.items():
            feats.append(self.encoders[name](x[:, idxs]))
        x = torch.cat(feats, dim=1)
        x = self.fusion(x)
        outs = []
        for name in self.decoders:
            outs.append(self.decoders[name](x))
        return torch.cat(outs, dim=1)

# ======================
# 加载模型
# ======================
model = SpatialAttentionResNet(input_vars, output_vars).to(DEVICE)
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

# ======================
# 读取单文件
# ======================
ds = xr.open_dataset(DATA_FILE)

# NaN 保护
for v in input_vars:
    if ds[v].isnull().any():
        ds[v] = ds[v].fillna(0)

X = ds[input_vars].to_array().transpose("time", "variable", "latitude", "longitude").values.astype(np.float32)

# ======================
# 推理
# ======================
preds = []
with torch.no_grad():
    for t in tqdm(range(X.shape[0]), desc="推理中"):
        x = torch.from_numpy(X[t:t+1]).to(DEVICE)
        y = model(x).cpu().numpy()[0]
        preds.append(y)

preds = np.stack(preds, axis=0)

# ======================
# 保存输出
# ======================
out_ds = xr.Dataset(
    {var: (("time", "latitude", "longitude"), preds[:, i]) for i, var in enumerate(output_vars)},
    coords={
        "time": ds.time.values,
        "latitude": ds.latitude.values,
        "longitude": ds.longitude.values
    }
)
out_ds.to_netcdf(OUT_FILE)
print(f"✅ Saved: {OUT_FILE}")


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
import numpy as np

# ========== 文件路径 ==========
files = {
    "pred_subseasonal_model_u": "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 脚本/pred_subseasonal_model_u_tp_20260831.nc",
    "SA_model": "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 脚本/33SpatialAttentionResNet_TP_20260831.nc",
    "UPU_NET": "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-tp/20260831-tp 脚本/33subseasonal_model_u-tp_best_acc_20260831.nc"
}

# 高对比色板
cmap = "turbo"

# 输出路径
output_dir = "./tp_maps"
os.makedirs(output_dir, exist_ok=True)

# ================= 读取所有文件，整理同名变量 =================
datasets = {name: xr.open_dataset(path) for name, path in files.items()}

# 取所有文件共有的变量（只考虑 (time, lat, lon) 维度）
all_vars = set(datasets[list(datasets.keys())[0]].data_vars)
for ds in datasets.values():
    all_vars &= set(ds.data_vars)
variables = [v for v in all_vars if datasets[list(datasets.keys())[0]][v].dims == ('time', 'latitude', 'longitude')]
variables = sorted(variables)  # 排序保证列顺序固定

# ================= 计算每个变量统一的 colorbar 范围 =================
vmins = {}
vmaxs = {}
for var in variables:
    vals = np.concatenate([datasets[name][var].isel(time=0).values.flatten() for name in datasets])
    vmins[var] = np.nanmin(vals)
    vmaxs[var] = np.nanmax(vals)

# ================= 绘图 =================
n_files = len(datasets)
n_vars = len(variables)

fig, axes = plt.subplots(nrows=n_files, ncols=n_vars, figsize=(5*n_vars, 4*n_files),
                         subplot_kw={'projection': ccrs.PlateCarree()})

# 如果只有一行或一列，axes可能是一维，需要处理成二维
if n_files == 1:
    axes = axes[np.newaxis, :]
if n_vars == 1:
    axes = axes[:, np.newaxis]

for row_i, (fname, ds) in enumerate(datasets.items()):
    for col_i, var in enumerate(variables):
        ax = axes[row_i, col_i]
        data = ds[var].isel(time=0)
        im = ax.pcolormesh(ds.longitude, ds.latitude, data,
                           cmap=cmap, shading='auto',
                           vmin=vmins[var], vmax=vmaxs[var])
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        if row_i == 0:
            ax.set_title(var, fontsize=12)
        if col_i == 0:
            ax.text(-0.08, 0.5, fname, va='center', ha='right', rotation=90,
                    transform=ax.transAxes, fontsize=12)

# 添加 colorbar，每列一个
for col_i, var in enumerate(variables):
    cax = fig.add_axes([0.1 + col_i/n_vars*0.8, 0.08, 0.8/n_vars, 0.02])  # [left, bottom, width, height]
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmins[var], vmax=vmaxs[var]))
    sm._A = []
    cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
    cbar.set_label('mm/day')

plt.tight_layout(rect=[0, 0.1, 1, 0.95])
plt.suptitle("2026-04-27", fontsize=16)
plt.show()
